<h1 style="text-align: center; font-size: 50px;"> Spam Detection with NLP (Natural Language Processing) MLflow Integration </h1>

Notebook Overview
- Start Execution
- User Constants
- Install and Import Libraries
- Configure Settings
- Verify Assets
- Logging Model to MLflow
- Fetching the Latest Model Version from MLflow
- Loading the Model and Running Inference

## Start Execution

In [1]:
import logging
import time

# Configure logger
logger: logging.Logger = logging.getLogger("register_model_logger")
logger.setLevel(logging.INFO)
logger.propagate = False  # Prevent duplicate logs from parent loggers

# Set formatter
formatter: logging.Formatter = logging.Formatter(
    fmt="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S"
)

# Configure and attach stream handler
stream_handler: logging.StreamHandler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

In [2]:
start_time = time.time()  

logger.info("Notebook execution started.")

2025-10-13 18:53:04 - INFO - Notebook execution started.


## User Constants

In [3]:
TEXT = "As a valued customer, I am pleased to advise you that following recent review of your Mob No. you are awarded with a $1500 Bonus Prize, call 09066364589"

##  Install and Import Libraries

In [4]:
%pip install -r ../requirements.txt --quiet

Note: you may need to restart the kernel to use updated packages.


In [5]:
# ------------------------ System Utilities ------------------------
import warnings
from pathlib import Path
import os
import yaml

# ------------------------ Data Manipulation ------------------------
import pandas as pd

# ------------------------ Text Preprocessing ------------------------
import string
import nltk
import sys
nltk.download('stopwords')
from nltk.corpus import stopwords
from types import SimpleNamespace
from sklearn.metrics import classification_report

# ------------------------ Machine Learning tools ------------------------
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

# ------------------------ MLflow for Experiment Tracking and Model Management ------------------------
import mlflow
from mlflow import MlflowClient
from mlflow.models.signature import ModelSignature
from mlflow.types.schema import Schema, ColSpec

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.mlflow import Logger

from src.utils import (
    load_config,
)

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Configure Settings

In [6]:
# Suppress Python warnings
warnings.filterwarnings("ignore")

In [ ]:
# ------------------------- Paths -------------------------
DATA_PATH = '/home/jovyan/datafabric/tutorial/spam_utf8.csv'
CONFIG_PATH = "../configs/config.yaml"
config = load_config(CONFIG_PATH)
DEMO_FOLDER = "../demo"

# ------------------------ MLflow Integration ------------------------
EXPERIMENT_NAME = "Spam_Detection_Experiment"
RUN_NAME = "Spam_Detection_Run"
MODEL_NAME = "Spam_Detection_Model"

## Verify Assets

In [ ]:
def log_asset_status(asset_path: str, asset_name: str, success_message: str, failure_message: str) -> None:
    """
    Logs the status of a given asset based on its existence.

    Parameters:
        asset_path (str): File or directory path to check.
        asset_name (str): Name of the asset for logging context.
        success_message (str): Message to log if asset exists.
        failure_message (str): Message to log if asset does not exist.
    """
    if Path(asset_path).exists():
        logger.info(f"{asset_name} is properly configured. {success_message}")
    else:
        logger.info(f"{asset_name} is not properly configured. {failure_message}")

log_asset_status(
    asset_path=DATA_PATH,
    asset_name="Spam data",
    success_message="",
    failure_message="Please create and download the required assets in your project on AI Studio."
)

log_asset_status(
    asset_path=DEMO_FOLDER,
    asset_name="Demo Folder",
    success_message="",
    failure_message="Please check if Demo folder was downloaded."
)

2025-10-13 18:53:07 - INFO - Spam data is properly configured. 
2025-10-13 18:53:07 - INFO - NLTK Path is not properly configured. Please check if NLTK was downloaded.
2025-10-13 18:53:07 - INFO - Demo Folder is properly configured. 


## Logging Model to MLflow

In [ ]:
# Configure MLflow tracking
mlflow.set_tracking_uri("/phoenix/mlflow")
mlflow.set_experiment(EXPERIMENT_NAME)

# Model signature for input/output schema
signature = ModelSignature(
    inputs  = Schema([ColSpec('string', 'text')]),
    outputs = Schema([ColSpec('string')])
)

# Log model using new Logger service
with mlflow.start_run() as run:
    
    # Log the model using models-from-code approach
    Logger.log_model(
        signature=signature,
        artifact_path=MODEL_NAME,
        config_path=CONFIG_PATH,
        data_file_path=DATA_PATH,
        demo_folder=DEMO_FOLDER
    )

    # Register the model
    model_uri = f"runs:/{run.info.run_id}/{MODEL_NAME}"
    mlflow.register_model(
        model_uri=model_uri,
        name=MODEL_NAME
    )

    logger.info(f"✅ Model registered successfully with run ID: {run.info.run_id}")

Registered model 'Spam_Detection_Model' already exists. Creating a new version of this model...
2025/10/13 18:53:13 WARNING mlflow.tracking._model_registry.fluent: Run with id 9e07b224e6dc47ad8057e372242b135c has no artifacts at artifact path 'Spam_Detection_Model', registering model based on models:/m-3bbf700e91d142abaaed5840790964f2 instead
Created version '5' of model 'Spam_Detection_Model'.
2025-10-13 18:53:14 - INFO - ✅ Model registered successfully with run ID: 9e07b224e6dc47ad8057e372242b135c


## Fetching the Latest Model Version from MLflow

In [10]:
# Initialize the MLflow client
client = MlflowClient()

# Retrieve the latest version of the "spam_detect_model" model 
model_metadata = client.get_latest_versions(MODEL_NAME, stages=["None"])
latest_model_version = model_metadata[0].version  # Extract the latest model version

# Fetch model information, including its signature
model_info = mlflow.models.get_model_info(f"models:/{MODEL_NAME}/{latest_model_version}")

# Print the latest model version and its signature
logger.info(f"Latest Model Version: {latest_model_version}")
logger.info(f"Model Signature: {model_info.signature}")

2025-10-13 18:53:16 - INFO - Latest Model Version: 5
2025-10-13 18:53:16 - INFO - Model Signature: inputs: 
  ['text': string (required)]
outputs: 
  [string (required)]
params: 
  None



## Loading the Model and Running Inference

In [11]:
model = mlflow.pyfunc.load_model(model_uri=f"models:/{MODEL_NAME}/{latest_model_version}")

# Define a sample text for testing
text_df = pd.DataFrame({'text': [TEXT]})

# Use the model to predict 
result = model.predict(text_df)
logger.info(result)

NLTK data directory not found at: /phoenix/mlflow/867553724460049213/models/m-3bbf700e91d142abaaed5840790964f2/artifacts/data/model_artifacts/nltk_data. Creating fallback.
2025-10-13 18:53:17 - INFO - ['spam']


In [12]:
end_time: float = time.time()
elapsed_time: float = end_time - start_time
elapsed_minutes: int = int(elapsed_time // 60)
elapsed_seconds: float = elapsed_time % 60

logger.info(f"⏱️ Total execution time: {elapsed_minutes}m {elapsed_seconds:.2f}s")
logger.info("✅ Notebook execution completed successfully.")

2025-10-13 18:53:17 - INFO - ⏱️ Total execution time: 0m 13.29s
2025-10-13 18:53:17 - INFO - ✅ Notebook execution completed successfully.


Built with ❤️ using [**Z by HP AI Studio**](https://zdocs.datascience.hp.com/docs/aistudio/overview).